# The Pokémon Company - PTCG AI Battle Challenge: Prism Training Pipeline

This notebook trains and packages **`prism`**: a **Belief-Conditioned Relational Actor-Critic** architecture for the Pokémon Trading Card Game (Bo3).

### Core Architecture Highlights:
1. **Composite Entity Tokenizer ($X \in \mathbb{R}^{B \times 64 \times 256}$)**: Maps 64 active game entities across static rules ($e_{\text{static}} \in \mathbb{R}^{128}$), dynamic LIFO evolution stacks and damage ($e_{\text{dynamic}} \in \mathbb{R}^{96}$), and zone coordinates ($e_{\text{zone}} \in \mathbb{R}^{32}$).
2. **Relational Set Transformer Backbone**: 4 Set Attention Blocks (SAB) enforcing permutation equivariance over bench and hand entities, with PMA_1 global pooling.
3. **Field-Inference Caching**: Decouples the $O(L \cdot N^2 \cdot d)$ backbone from the $O(N \cdot d)$ pointer decoder, achieving a **43x speedup** on multi-step card resolution sequences.
4. **Dual-Horizon Value Critic**: Evaluates game equity $V_{\text{game}}$ and match equity $V_{\text{match}}$, driving strategic concession under tournament clock pressure.
5. **3-Segment Deck & Bayesian Belief Tracker**: Hypergeometric distributions over hidden zones and real-time Dirichlet/Multinomial archetype inference.

## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install -q kaggle-environments torch numpy scipy
import os
import sys
import time
import torch
import numpy as np

print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")

## 2. Load Prism Codebase & Modules
Clones the official GitHub repository into the Kaggle environment and imports Prism modules.

In [ ]:
import os
import sys

# Clone repository if running in a clean Kaggle container
if not os.path.exists("models") and not os.path.exists("prism/models"):
    !git clone https://github.com/DLC-17/prism.git
if os.path.exists("prism/models"):
    %cd prism

if "." not in sys.path:
    sys.path.insert(0, ".")

from models.actor_critic import PrismActorCritic
from engine.cabt_adapter import CabtObservationAdapter
from engine.python_fallback.card_db import CardCatalog
from engine.hydrapple_catalog import register_hydrapple_archetype, create_hydrapple_deck_list
from inference.cache_manager import FieldInferenceEngine
from kaggle.train_kaggle import train_prism

print("All Prism modules loaded successfully.")

## 3. Train Prism Actor-Critic (Multi-Task Learning)
Trains `MacroActionHead`, `EntityPointerHead`, and `DualHorizonCritic` jointly.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

train_prism(
    epochs=5,
    steps_per_epoch=50,
    batch_size=32,
    lr=3e-4,
    device=device,
    output_dir="./checkpoints",
)

## 4. Run Latency Budget Verification (2 vCPU Budget)
Verifies that turn calculation time stays well under the strict 50 ms budget across 20 atomic sub-actions.

In [ ]:
from tests.bench_turn_latency import benchmark_components

benchmark_components()

## 5. Package Competition Submission Bundle
Creates `submission.tar.gz` containing `main.py`, `deck.csv`, `models/`, `engine/`, `inference/`, and trained weights `prism_model.pt`.

In [ ]:
from scripts.package_submission import package_submission

package_submission()

!ls -lh submission.tar.gz

## 6. Validate Submission Agent
Tests that `main.agent` can be invoked cleanly by Kaggle environments.

In [ ]:
from kaggle.submission.main import agent

# Test mock observation
mock_obs = {
    "select": {
        "type": "MAIN",
        "option": [{"type": "ATTACK", "area": "ACTIVE", "index": 0}, {"type": "END"}]
    },
    "current": {
        "turn": 1,
        "yourIndex": 0,
        "players": [{"active": [None], "bench": [], "hand": [], "prize": []}, {"active": [None], "bench": []}]
    },
    "logs": []
}

action = agent(mock_obs)
print(f"Agent chosen action option index: {action}")
assert isinstance(action, list) and len(action) == 1
print("SUBMISSION VALIDATION PASSED!")